# 02 - Preprocessing
Load 20-50 images, apply resize, normalization, and cleaning.

In [ ]:
from pathlib import Path
import sys
import cv2

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_loader import load_metadata, assign_group_labels, get_channel_columns, load_multichannel_image
from preprocess import resize_image, normalize_image, clean_image

meta_path = PROJECT_ROOT / 'data' / 'raw' / 'metadata' / 'BBBC021_v1_image.csv'
images_dir = PROJECT_ROOT / 'data' / 'raw' / 'images'
out_resize = PROJECT_ROOT / 'data' / 'processed' / 'resized'
out_norm = PROJECT_ROOT / 'data' / 'processed' / 'normalized'
out_clean = PROJECT_ROOT / 'data' / 'processed' / 'cleaned'
out_resize.mkdir(parents=True, exist_ok=True)
out_norm.mkdir(parents=True, exist_ok=True)
out_clean.mkdir(parents=True, exist_ok=True)

meta = assign_group_labels(load_metadata(meta_path))
channel_cols = get_channel_columns(meta)
sample_df = meta.head(30).copy()
print('Processing', len(sample_df), 'images')

In [ ]:
for idx, row in sample_df.iterrows():
    img_id = f'image_{idx:05d}'
    fused, _ = load_multichannel_image(row, images_dir, channel_cols)

    resized = resize_image(fused, (512, 512))
    normalized = normalize_image(resized)
    cleaned = clean_image(normalized)

    cv2.imwrite(str(out_resize / f'{img_id}.png'), resized)
    cv2.imwrite(str(out_norm / f'{img_id}.png'), normalized)
    cv2.imwrite(str(out_clean / f'{img_id}.png'), cleaned)

print('Saved preprocessed images to processed folders.')